# ROSTR.cc → Google Sheet — single-notebook demo

**The story:** keep a Google Sheet of artists. Every night, look each artist up on rostr.cc and fill in `Agency / Agent / Management / Management Contact` for any row where those columns are blank. Manual edits in the sheet are preserved — we only write empty cells.

**The whole pipeline is two HTTP APIs.** No Spark, no Databricks Connect, no headless browser. The Databricks angle is just “this runs as a scheduled job in the workspace.”

**What's in this notebook:**
1. Install `requests` + `google-auth`
2. Hard-code credentials and the sheet ID
3. Define rostr.cc + Google Sheets helpers (everything inline so you can read it top to bottom)
4. Log in, read the sheet, look up each artist, write the missing cells back
5. Print a one-line summary

_Sheet schema this expects:_

| A | B | C | D | E | F |
|---|---|---|---|---|---|
| Artist | Agency | Agent Name | Management | Management Contact | Last Updated |

> Google Sheets auth here is **OAuth user creds** — the notebook authenticates as you, so any sheet you can edit, the notebook can edit. No service account or sheet-sharing step. See the appendix at the bottom for the one-time setup that gets you three OAuth strings.

## 0. Install dependencies

Both libraries are pure-Python and tiny. Restart the Python interpreter after install so the kernel picks them up.

In [0]:
%pip install --quiet requests google-auth
dbutils.library.restartPython()

## 1. Configuration

Hard-coded for this demo. In production swap each value for `dbutils.secrets.get("scope", "key")` — the rest of the notebook doesn't change.

The three OAuth values come from the one-time local browser dance — see the appendix below.

In [0]:
# --- rostr.cc account ---
ROSTR_USERNAME = "YOUR_ROSTR_EMAIL@example.com"
ROSTR_PASSWORD = "YOUR_ROSTR_PASSWORD"

# --- Google Sheet ---
GOOGLE_SHEET_ID  = "YOUR_SHEET_ID"   # the bit between /d/ and /edit in the sheet URL
GOOGLE_SHEET_TAB = "Sheet1"

# --- Google OAuth user credentials ---
# Get these from the one-time local OAuth dance — see the appendix at the bottom of
# this notebook. Auth runs as your Google identity, so any sheet you can edit, this
# notebook can edit. No SA + no sheet-sharing required.
GOOGLE_OAUTH_CLIENT_ID     = "YOUR_CLIENT_ID.apps.googleusercontent.com"
GOOGLE_OAUTH_CLIENT_SECRET = "YOUR_CLIENT_SECRET"
GOOGLE_OAUTH_REFRESH_TOKEN = "YOUR_REFRESH_TOKEN"

## 2. rostr.cc helpers

The public rostr.cc *web* login is gated by Cloudflare Turnstile, but the underlying JSON API only needs an email + password — fraud checks are enforced at the web origin only. So we POST to `/v1/auth/rostr` once for a session cookie, then GET `/v1/artist/{slug}/team/{role}` for each artist.

`slugify("Olivia Rodrigo") == "oliviarodrigo"`, which matches rostr.cc's URL handles.

In [0]:
import re, json, unicodedata, datetime as dt
import requests

# This is the base URL we talk to. rostr.cc is a music industry database
# that tracks which agents/managers represent which artists.
ROSTR_API = "https://api.rostr.cc"


def slugify(name: str) -> str:
    # rostr.cc URLs look like rostr.cc/artist/oliviarodrigo
    # So we need to turn "Olivia Rodrigo" into "oliviarodrigo"
    # and handle accented names like "RÜFÜS DU SOL" -> "rufusdusol"
    ascii_name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]", "", ascii_name.lower())


def rostr_login(username: str, password: str) -> requests.Session:
    # Logs into rostr.cc and gives us back a "session" (think of it like
    # staying logged in so we don't have to re-enter the password every time).
    #
    # The website normally has a captcha, but the behind-the-scenes API doesn't
    # check for one — we just need email + password. We pretend to be a browser
    # so their security system doesn't block us.
    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/json",
        "Origin": "https://www.rostr.cc",
        "Referer": "https://www.rostr.cc/",
        "Content-Type": "application/json",
    })

    resp = session.post(
        f"{ROSTR_API}/v1/auth/rostr",
        json={"email": username, "password": password},
        timeout=20,
    )
    # 201 means "success, you're logged in". Anything else = wrong password or blocked.
    if resp.status_code != 201:
        raise RuntimeError(f"rostr login failed ({resp.status_code}): {resp.text[:200]}")
    return session


def rostr_lookup(session: requests.Session, artist_name: str) -> dict:
    # This is the main function. Give it an artist name like "Billie Eilish"
    # and it returns a dictionary like:
    #   {"agency": "WME", "agent": "John Smith",
    #    "management": "Darkroom", "management_contact": "jane@mgmt.com"}
    # If the artist isn't on rostr, all values come back as empty strings.
    slug = slugify(artist_name)
    result = {"agency": "", "agent": "", "management": "", "management_contact": ""}

    def _fetch_team(role):
        # Asks rostr for one type of team member. role is either "AGENCY" or "MANAGEMENT".
        # If the artist doesn't exist on rostr, we just get a 404 — that's fine, not an error.
        resp = session.get(f"{ROSTR_API}/v1/artist/{slug}/team/{role}", timeout=20)
        if resp.status_code == 404:
            return []
        resp.raise_for_status()
        data = resp.json()
        return data if isinstance(data, list) else []

    def _flatten(entries):
        # rostr sends back data in a complicated nested format like:
        #   [{"company": {"name": "WME"},
        #     "team": [{"people": [{"name": "John", "email": "j@wme.com"}]}]}]
        #
        # All we want is simple flat lists:
        #   companies = ["WME", "CAA"]
        #   people    = ["John Smith", "Jane Doe"]
        #   contacts  = ["j@wme.com", "jane@caa.com"]
        companies, people, contacts = [], [], []

        for entry in entries:
            co = ((entry.get("company") or {}).get("name") or "").strip()
            if co and co not in companies:
                companies.append(co)

            # Dig into the nested structure to find actual people
            for group in entry.get("team") or []:
                for person in group.get("people") or []:
                    name = (person.get("name") or "").strip()
                    email = (person.get("email") or "").strip()
                    phone = (person.get("phone") or "").strip()

                    if name and name not in people:
                        people.append(name)

                    # For contact info: use email if available, otherwise phone, otherwise name
                    contact = email or phone or name
                    if contact:
                        contacts.append(contact)

        return companies, people, contacts

    # Look up both agency and management for this artist
    agency_cos, agency_ppl, _ = _flatten(_fetch_team("AGENCY"))
    mgmt_cos, mgmt_ppl, mgmt_contacts = _flatten(_fetch_team("MANAGEMENT"))

    # If an artist has multiple agencies (rare), join them with " / "
    result["agency"] = " / ".join(agency_cos)
    result["agent"] = " / ".join(agency_ppl)
    result["management"] = " / ".join(mgmt_cos)
    # Just grab the first management contact (usually the main manager's email)
    result["management_contact"] = mgmt_contacts[0] if mgmt_contacts else ""
    return result

## 3. Google Sheets helpers

Plain Sheets v4 REST. Three operations:

- `read_sheet`   — pull all rows in the tab as a list-of-lists
- `needs_enrichment` — a row needs help when any of cols B–E is blank
- `write_cells`  — batch-update only the cells we want to fill

Auth is OAuth user creds. `google-auth` refreshes the access token automatically when it expires (every ~1h) using the refresh token. We only write **blank** cells, so manual edits in the sheet are never overwritten.

In [0]:
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import AuthorizedSession

SHEETS_API = "https://sheets.googleapis.com/v4/spreadsheets"
SHEETS_SCOPE = ["https://www.googleapis.com/auth/spreadsheets"]

# These match the columns in your Google Sheet:
# A=Artist, B=Agency, C=Agent, D=Management, E=Contact, F=Last Updated
COL_ARTIST, COL_AGENCY, COL_AGENT, COL_MGMT, COL_CONTACT, COL_UPDATED = 0, 1, 2, 3, 4, 5


def sheets_session(client_id: str, client_secret: str, refresh_token: str) -> AuthorizedSession:
    # Sets up a connection to Google Sheets using your OAuth credentials.
    # Think of it like logging into Google — the refresh token means we don't
    # need to open a browser every time, it just auto-renews.
    creds = Credentials(
        token=None,
        refresh_token=refresh_token,
        client_id=client_id,
        client_secret=client_secret,
        token_uri="https://oauth2.googleapis.com/token",
        scopes=SHEETS_SCOPE,
    )
    return AuthorizedSession(creds)


def read_sheet(http, sheet_id: str, tab: str) -> list:
    # Pulls all rows from the Google Sheet tab. Returns a list of rows,
    # where each row is a list of cell values like ["Billie Eilish", "WME", ...].
    r = http.get(f"{SHEETS_API}/{sheet_id}/values/{tab}!A1:Z1000")
    r.raise_for_status()
    return r.json().get("values", [])


def needs_enrichment(row) -> bool:
    # Checks if a row is missing team info (any of Agency/Agent/Management/Contact is blank).
    # We skip rows with no artist name — those are just empty rows at the bottom.
    row = row + [""] * (6 - len(row))  # pad short rows so we don't crash on missing columns
    if not (row[COL_ARTIST] or "").strip():
        return False
    return not all((row[c] or "").strip() for c in (COL_AGENCY, COL_AGENT, COL_MGMT, COL_CONTACT))


def write_cells(http, sheet_id: str, tab: str, updates: list) -> int:
    # Writes specific cells back to the Google Sheet.
    # updates is a list of (row_number, column_number, value) — we only write
    # into cells that are currently blank, so we never overwrite manual edits.
    # Returns how many cells were written.
    if not updates:
        return 0
    body = {
        "valueInputOption": "USER_ENTERED",
        "data": [
            {"range": f"{tab}!{chr(ord('A') + c)}{r}", "values": [[v]]}
            for r, c, v in updates
        ],
    }
    resp = http.post(f"{SHEETS_API}/{sheet_id}/values:batchUpdate", json=body)
    resp.raise_for_status()
    return resp.json().get("totalUpdatedCells", 0)

## 4. Log in and pull the sheet

Both `rostr_login` and `sheets_session` are one-shot — we hold the resulting authenticated sessions and reuse them below.

In [0]:
# Log into both services (rostr + Google Sheets) — we reuse these connections below
rostr_http  = rostr_login(ROSTR_USERNAME, ROSTR_PASSWORD)
sheets_http = sheets_session(GOOGLE_OAUTH_CLIENT_ID, GOOGLE_OAUTH_CLIENT_SECRET, GOOGLE_OAUTH_REFRESH_TOKEN)

# Pull all rows from the sheet
rows = read_sheet(sheets_http, GOOGLE_SHEET_ID, GOOGLE_SHEET_TAB)
header, data = rows[0], rows[1:]  # first row is column headers, rest is data

# Figure out which rows are missing team info and need to be looked up
# i+2 because: row 1 is the header, Python counts from 0, so data row 0 = sheet row 2
to_enrich = [(i + 2, r) for i, r in enumerate(data) if needs_enrichment(r)]

print(f"Total rows : {len(data)}")
print(f"To enrich  : {len(to_enrich)}")
for row_num, r in to_enrich:
    print(f"  row {row_num:>3}: {(r + ['']*6)[COL_ARTIST]}")

## 5. Look each artist up on rostr.cc

One HTTP round-trip per artist. We compute the cells we want to write — only filling **blanks** — and stamp `Last Updated` for any row we touched.

Nothing is sent to Google yet — the next cell does the write.

In [0]:
# Timestamp we'll put in the "Last Updated" column for any row we touch
stamp = dt.datetime.now(dt.timezone.utc).replace(second=0, microsecond=0).isoformat()

updates = []   # collects all the cell writes we want to make (row, column, value)
resolved, failed = 0, 0

for row_num, row in to_enrich:
    row = row + [""] * (6 - len(row))  # pad so we don't crash on short rows
    artist = row[COL_ARTIST].strip()

    # Try to look up this artist on rostr — if it fails (network error, etc), skip them
    try:
        team = rostr_lookup(rostr_http, artist)
    except Exception as e:
        print(f"  \u2717 {artist:<30s} \u2014 {e}")
        failed += 1
        continue

    # For each team field, only write it if the sheet cell is currently blank
    # AND rostr actually had data for it. This way we never overwrite manual edits.
    wrote_any = False
    for col, key in [
        (COL_AGENCY,  "agency"),
        (COL_AGENT,   "agent"),
        (COL_MGMT,    "management"),
        (COL_CONTACT, "management_contact"),
    ]:
        if not (row[col] or "").strip() and team[key]:
            updates.append((row_num, col, team[key]))
            wrote_any = True

    # If we wrote anything for this artist, stamp the "Last Updated" column too
    if wrote_any:
        updates.append((row_num, COL_UPDATED, stamp))
    print(f"  \u2713 {artist:<30s} -> agency={team['agency']!r}  mgmt={team['management']!r}")
    resolved += 1

print(f"\nResolved {resolved}/{len(to_enrich)} artists \u2014 will write {len(updates)} cells.")

## 6. Write back to the sheet

Single batched call to `values:batchUpdate`. Re-running the notebook is idempotent: rows that are already fully populated are skipped in step 4, so we only ever write into blanks.

In [0]:
# Send all the updates to Google Sheets in one batch.
# This is the only place we actually write to the sheet.
n_written = write_cells(sheets_http, GOOGLE_SHEET_ID, GOOGLE_SHEET_TAB, updates)

sheet_url = f"https://docs.google.com/spreadsheets/d/{GOOGLE_SHEET_ID}/edit"
print(f"Wrote {n_written} cells to {sheet_url}")

## Summary

That's the whole demo. Refresh the Google Sheet to see the cells fill in. Re-run the notebook on the same sheet and step 4 will show `To enrich : 0` — it's idempotent.

**To run this nightly:**
1. Set up a scheduled task (cron job, GitHub Action, whatever you use) that runs this notebook once a day.
2. No special infrastructure needed — it's just Python + two HTTP APIs.

**For real production use:** swap each hardcoded credential for environment variables or a secrets manager. The rest of the notebook is unchanged.

In [0]:
# Final summary — just prints what happened so you can eyeball it in the job logs
print("=" * 60)
print(f"  Considered      : {len(to_enrich)} rows")
print(f"  Resolved        : {resolved}")
print(f"  Failed lookups  : {failed}")
print(f"  Cells written   : {n_written}")
print(f"  Stamp           : {stamp}")
print("=" * 60)

---

## Appendix — one-time setup to get the three OAuth strings

Do this **once, locally on your laptop**. The output is three small strings that you paste into the Configuration cell at the top.

### 1. GCP setup (≈3 min)
1. Open <https://console.cloud.google.com> and pick or create a project.
2. **APIs & Services → Library** → search **Google Sheets API** → **Enable**.
3. **APIs & Services → OAuth consent screen**:
    - User type: **Internal** (only your org's users — refresh tokens never auto-expire).
    - App name: `rostr-demo`. Support + developer email: your work email. Defaults for the rest.
4. **APIs & Services → Credentials → + Create credentials → OAuth client ID**:
    - Application type: **Desktop app**.  Name: `rostr-demo-local`.
    - **Download JSON** when prompted. Save it next to wherever you'll run the script below as `client_secret.json`.

### 2. Run the OAuth dance locally (≈1 min)
On your laptop (not in this notebook):

```bash
pip install google-auth-oauthlib
```

```python
# oauth_dance.py — run once
from google_auth_oauthlib.flow import InstalledAppFlow
flow = InstalledAppFlow.from_client_secrets_file(
    'client_secret.json',
    scopes=['https://www.googleapis.com/auth/spreadsheets'],
)
creds = flow.run_local_server(port=0)
print(f'CLIENT_ID     = {creds.client_id!r}')
print(f'CLIENT_SECRET = {creds.client_secret!r}')
print(f'REFRESH_TOKEN = {creds.refresh_token!r}')
```

A browser opens → log in with your work account → grant the Sheets scope → terminal prints three strings.

### 3. Paste into the Configuration cell
Copy the three values into `GOOGLE_OAUTH_CLIENT_ID / GOOGLE_OAUTH_CLIENT_SECRET / GOOGLE_OAUTH_REFRESH_TOKEN` at the top. That's it — the refresh token works from anywhere, including scheduled runs of this notebook.